In [1]:
# !pip install torchvision

In [2]:
import torch
import os
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms 
from PIL import Image  # PIL => Python imaging Library using this we can load image and convert into rgb

## Load Dataset

In [3]:
# images load => transform => dataset of all images
class ImageProcessor():
    def __init__(self, root_dir_path, transformations=None):
        self.root_dir_path = root_dir_path
        self.transformations = transformations

        # list of paths for all images
        self.all_img_paths= [os.path.join(root_dir_path, img) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self, idx):
        img_path = self.all_img_paths[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)

        return img

In [4]:
root_dir_path = "./img_align_celeba/"
transformations = transforms.Compose(
    [
        transforms.CenterCrop(178),  # 178 * 218 => 178 * 178
        transforms.Resize(64),  # 64 * 64
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5))  # [-1,1]
    ]
)

In [5]:
dataset= ImageProcessor(root_dir_path, transformations)
print(f"loaded {len(dataset)} images")

loaded 202599 images


In [6]:
dataloader = DataLoader(dataset, batch_size=128, shuffle= True)

## Generator Network

In [7]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [8]:
class Generator(nn.Module):
    def __init__(self, z_dim = 100, img_channels = 3): # 3 is for RGB
        super(Generator, self).__init__()

        # fully connected (Dense) Layers
        self.model = nn.Sequential(
            nn.Linear(z_dim, 256), # 100 => 256
            nn.ReLU(),

            nn.Linear(256, 512),
            nn.ReLU(),

            nn.Linear(512, 1024),
            nn.ReLU(),
            
            nn.Linear(1024, 64 * 64 * img_channels),
            nn.Tanh()  # [-1,1]
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), 3, 64, 64)
        return img

## Discriminator Network

In [9]:
class Discriminator(nn.Module):
    def __init__(self, img_channels = 3): # 3 is for RGB
        super(Discriminator, self).__init__()

        # fully connected (Dense) Layers
        self.model = nn.Sequential(
            # fake image generated dimensions are => 64 x 64 x 3 x batch_size   becuase of batch size it became 4 dimensional output
            nn.Flatten(),  # 4D tensors convert into 1D
            
            nn.Linear(img_channels * 64 * 64, 1024), # 
            nn.LeakyReLU(0.2, inplace= True),

            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace= True),

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace= True),
            
            nn.Linear(256, 1),
            nn.Sigmoid()  # Probability of real or fake
        )

    def forward(self, img):
        return self.model(img)

In [10]:
GAN_loss = nn.BCELoss()

generator = Generator()
g_optimizer = optim.Adam(generator.parameters(), lr=0.0002 , betas=(0.5, 0.999))

discriminator = Discriminator()
d_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))

In [11]:
# device
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = "cuda not avail"
print(f"device is {device}")

device is cuda


In [12]:
generator = generator.to(device)
discriminator = discriminator.to(device)

In [13]:
print(f"{ 70 * '-'}\Generator :-\n{ 70 * '-'} \n{generator} \n{ 70 * '-'}")
print(f"Discriminator :-\n{ 70 * '-'} \n{discriminator}")

----------------------------------------------------------------------\Generator :-
---------------------------------------------------------------------- 
Generator(
  (model): Sequential(
    (0): Linear(in_features=100, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=1024, bias=True)
    (5): ReLU()
    (6): Linear(in_features=1024, out_features=12288, bias=True)
    (7): Tanh()
  )
) 
----------------------------------------------------------------------
Discriminator :-
---------------------------------------------------------------------- 
Discriminator(
  (model): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=12288, out_features=1024, bias=True)
    (2): LeakyReLU(negative_slope=0.2, inplace=True)
    (3): Linear(in_features=1024, out_features=512, bias=True)
    (4): LeakyReLU(negative_slope=0.2, inplace=True)
    (5): Line

# Training The Gan

In [14]:
def train(generator, discriminator, dataloader, epochs=10):

    for epoch in range(epochs):
        for i, imgs in enumerate(dataloader):
            real_imgs = imgs.to(device)
            batch_size = real_imgs.size(0)

            # creates real imgs labels & fake imgs labels
            real_labels = torch.ones(batch_size, 1).to(device) # [1, 1, 1....]
            fake_labels = torch.zeros(batch_size, 1).to(device) # [0, 0, 0....]

            # Train the Discriminator
            d_optimizer.zero_grad()

            fake_imgs = generator(torch.randn(batch_size, 100).to(device))

            real_loss = GAN_loss(discriminator(real_imgs), real_labels)
            fake_loss = GAN_loss(discriminator(fake_imgs.detach()), fake_labels)

            d_loss = (real_loss + fake_loss) / 2

            d_loss.backward()
            d_optimizer.step()

            # Train the Generator
            g_optimizer.zero_grad()

            g_loss = GAN_loss(discriminator(fake_imgs), real_labels)

            g_loss.backward()
            g_optimizer.step()

            if i % 50 == 0:
                print(f"for epoch: {epoch+1}/{epochs}... batch: {i+1}... G-loss:{g_loss}.... D-loss: {d_loss}")

        # save generated imgs for each epoch
        save_generated_images(generator, epoch, device) 

In [15]:
import matplotlib.pyplot as plt
import numpy as np
import torchvision

In [18]:
def save_generated_images(generator, epoch, device, num_imgs=8):
    z = torch.randn(num_imgs, 100).to(device)
    generated_imgs = generator(z).detach().cpu()

    grid = torchvision.utils.make_grid(generated_imgs, nrow= 4, normalize= True)
        
    plt.imshow(np.transpose(grid, (1,2, 0)))
    plt.title(f"epoch {epoch +1 }")
    plt.axis("off")
    plt.show()

In [19]:
train(generator, discriminator, dataloader)